# XAUUSD LLM Forecaster — Colab / Kaggle Pipeline

**Before running:**
1. Enable GPU:
   - **Colab**: Runtime → Change runtime type → GPU (T4 or better)
   - **Kaggle**: Settings panel → Accelerator → GPU P100, and turn **Internet On**
2. Run all cells top-to-bottom — no uploads needed, everything is cloned from GitHub.

## 0. Setup — Mount Drive, Install Dependencies

In [ ]:
import os, sys

REPO_URL = 'https://github.com/211abhi/xauusd-llm-forecaster.git'
CODE_DIR = '/kaggle/working/xauusd-llm-forecaster' if os.path.exists('/kaggle') else '/content/xauusd-llm-forecaster'

if not os.path.isdir(CODE_DIR):
    !git clone {REPO_URL} {CODE_DIR}
else:
    !git -C {CODE_DIR} pull

%cd {CODE_DIR}
sys.path.insert(0, CODE_DIR)
os.environ['PYTHONPATH'] = CODE_DIR
os.environ['CODE_DIR'] = CODE_DIR

for d in ['data/raw', 'data/processed', 'data/splits',
          'checkpoints/encoder', 'checkpoints/soft_prompt', 'checkpoints/pred_head',
          'logs']:
    os.makedirs(os.path.join(CODE_DIR, d), exist_ok=True)

env = 'Kaggle' if os.path.exists('/kaggle') else 'Colab'
print(f'Environment : {env}')
print(f'Working dir : {os.getcwd()}')

In [ ]:
import os
if not os.path.exists('/kaggle'):
    !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers>=4.40.0 datasets ta cma scikit-learn pyyaml tqdm matplotlib seaborn

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
import yaml, os

CODE_DIR = os.environ.get('CODE_DIR', '/content/xauusd-llm-forecaster')

# ── User inputs ────────────────────────────────────────────────
output_steps = int(input('Forecast steps? (1 step = 30 min | 8=4h  16=8h  48=24h) [default 16]: ') or '16')
print(f'  → forecasting {output_steps} steps = {output_steps * 30 / 60:.1f} hours ahead')

cfg = {
    'project': {
        'name': 'xauusd-llm-forecaster',
        'seed': 42,
        'device': device,
        'log_dir': f'{CODE_DIR}/logs/',
        'checkpoint_dir': f'{CODE_DIR}/checkpoints/',
    },
    'data': {
        'raw_path': f'{CODE_DIR}/data/raw/XAU_30m_data.csv',
        'processed_dir': f'{CODE_DIR}/data/processed/',
        'splits_dir': f'{CODE_DIR}/data/splits/',
        'datetime_col': 'datetime',
        'feature_cols': ['open', 'high', 'low', 'close', 'volume'],
        'derived_features': ['returns', 'log_returns', 'hl_range', 'body_ratio'],
        'indicators': {'atr_period': 14, 'rsi_period': 14, 'ema_fast': 20, 'ema_slow': 50},
        'split_ratios': {'train': 0.70, 'val': 0.15, 'test': 0.15},
        'normalization': 'rolling_minmax',
        'norm_window': 500,
    },
    'regime_labeling': {
        'rsi_high': 55, 'rsi_low': 45, 'rsi_mid_low': 40, 'rsi_mid_high': 60,
        'ranging_atr_mult': 0.3, 'atr_spike_mult': 1.5, 'hl_range_spike_mult': 1.5,
        'breakout_lookback': 20, 'volume_spike_mult': 1.5,
    },
    'tokenizer': {
        'window_size': 96, 'patch_size': 16, 'n_patches': 6,
        'n_features': 9, 'patch_dim': 144,
        'train_stride': 1, 'val_stride': 8,
    },
    'encoder': {
        'embed_dim': 256, 'n_heads': 8, 'n_layers': 4, 'ffn_dim': 512,
        'dropout': 0.1, 'max_seq_len': 7, 'output_dim': 256,
        'checkpoint_path': f'{CODE_DIR}/checkpoints/encoder/best_encoder.pt',
    },
    'alignment': {
        'temperature': 0.07,
        'projection_dim': 1024,
        'n_regimes': 5,
        'regime_labels': ['TRENDING_UP', 'TRENDING_DOWN', 'RANGING', 'VOLATILE', 'BREAKOUT'],
        'n_negatives': 4,
    },
    'encoder_training': {
        'epochs': 20, 'batch_size': 64, 'lr': 1e-4, 'weight_decay': 1e-5,
        'warmup_steps': 200, 'lr_schedule': 'cosine', 'patience': 10, 'val_every': 1,
    },
    'llm': {
        'model_name': 'gpt2-medium',
        'hidden_dim': 1024,
        'freeze_all': True,
        'hidden_state_layer': -1, 'hidden_state_position': 0,
        'cache_dir': f'{CODE_DIR}/.cache/llm',
    },
    'soft_prompt': {
        'n_tokens': 32,
        'token_dim': 1024,
        'init_std': 0.01,
        'checkpoint_path': f'{CODE_DIR}/checkpoints/soft_prompt/best_soft_prompt.npy',
    },
    'cmaes': {
        'subspace_dim': 500, 'popsize': 20, 'sigma0': 0.1, 'maxiter': 100,
        'early_stop_patience': 30, 'fitness_metric': 'val_mae',
        'eval_batch_size': 32, 'max_eval_batches': 32,
    },
    'prediction_head': {
        'input_dim': 1024,
        'hidden_dim': 256, 'output_steps': output_steps, 'dropout': 0.1,
        'checkpoint_path': f'{CODE_DIR}/checkpoints/pred_head/best_pred_head.pt',
    },
    'pred_head_training': {
        'epochs': 30, 'batch_size': 32, 'lr': 1e-3, 'weight_decay': 1e-5,
        'patience': 10, 'loss': 'mse',
    },
    'evaluation': {
        'metrics': ['mae', 'rmse', 'directional_accuracy'],
        'per_regime': True, 'rolling_window_months': 1,
    },
}

cfg_path = f'{CODE_DIR}/configs/base_config.yaml'
os.makedirs(os.path.dirname(cfg_path), exist_ok=True)
with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f'Config written → {cfg_path}')
print(f'  device        : {device}')
print(f'  output_steps  : {output_steps} ({output_steps * 30 / 60:.1f} hours)')
print(f'  llm hidden_dim: {cfg["llm"]["hidden_dim"]}')

## Phase 1 — Data Preparation

Cleans raw OHLCV, adds indicators, normalizes, and saves train/val/test splits.

In [ ]:
!python {CODE_DIR}/scripts/01_prepare_data.py --config {CODE_DIR}/configs/base_config.yaml

In [ ]:
# Quick sanity check
import pandas as pd
df = pd.read_csv('data/processed/xau_processed.csv', parse_dates=['datetime'])
print(f'Processed rows: {len(df)}')
print(df.describe().round(3))

## Phase 1b — Regime Labeling

Assigns rule-based market regime labels (TRENDING_UP / DOWN / RANGING / VOLATILE / BREAKOUT).

In [ ]:
!python {CODE_DIR}/scripts/02_label_regimes.py --config {CODE_DIR}/configs/base_config.yaml

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv('data/processed/xau_with_regimes.csv', parse_dates=['datetime'])
counts = df['regime'].value_counts()
counts.plot(kind='bar', figsize=(8, 4), title='Regime Distribution (per candle)', color='steelblue')
plt.tight_layout()
plt.show()
print(counts)

## Phase 2 — Train TS Encoder (Contrastive)

Trains the 1D-Conv + Transformer encoder against frozen LLM text embeddings via InfoNCE loss.

⏱ ~20–40 min on T4 depending on dataset size.

In [ ]:
!python {CODE_DIR}/scripts/03_train_encoder.py --config {CODE_DIR}/configs/base_config.yaml

In [ ]:
from pathlib import Path
import os
CODE_DIR = os.environ.get('CODE_DIR', '/content/xauusd-llm-forecaster')
ckpt = Path(f'{CODE_DIR}/checkpoints/encoder/best_encoder.pt')
print('Encoder checkpoint saved:', ckpt.exists())
if ckpt.exists():
    print(f'Size: {ckpt.stat().st_size / 1e6:.1f} MB')

In [ ]:
# Save encoder checkpoint to GitHub so you never need to retrain
# Paste a GitHub Personal Access Token (Settings → Developer settings → PAT → classic, repo scope)
from getpass import getpass
from pathlib import Path
import os

CODE_DIR = os.environ.get('CODE_DIR', '/content/xauusd-llm-forecaster')
env_name = 'Kaggle' if os.path.exists('/kaggle') else 'Colab'

ckpt = Path(f'{CODE_DIR}/checkpoints/encoder/best_encoder.pt')
if ckpt.exists():
    token = getpass('GitHub PAT (hidden): ')
    !git -C {CODE_DIR} config user.email "training@env"
    !git -C {CODE_DIR} config user.name "{env_name}"
    !git -C {CODE_DIR} remote set-url origin https://{token}@github.com/211abhi/xauusd-llm-forecaster.git
    !git -C {CODE_DIR} add -f checkpoints/encoder/best_encoder.pt
    !git -C {CODE_DIR} commit -m "Add trained encoder checkpoint"
    !git -C {CODE_DIR} push origin main
    print('Encoder checkpoint pushed to GitHub.')
else:
    print('No checkpoint found — training may have failed.')

## Phase 3 — CMA-ES Soft Prompt Optimization

Optimizes the (32, 768) soft prompt prefix in a 500-dim subspace using evolutionary search.

⏱ ~1–3 hours on T4 (300 generations × 20 candidates). Reduce `maxiter` in config to speed up.

**To speed up:** Edit `configs/base_config.yaml` → `cmaes.maxiter: 50` for a quick test run.

In [ ]:
!python {CODE_DIR}/scripts/04_optimize_soft_prompt.py --config {CODE_DIR}/configs/base_config.yaml

In [ ]:
import numpy as np, os
from pathlib import Path
CODE_DIR = os.environ.get('CODE_DIR', '/content/xauusd-llm-forecaster')
sp_path = Path(f'{CODE_DIR}/checkpoints/soft_prompt/best_soft_prompt.npy')
print('Soft prompt saved:', sp_path.exists())
if sp_path.exists():
    arr = np.load(str(sp_path))
    print(f'Shape: {arr.shape}  mean={arr.mean():.4f}  std={arr.std():.4f}')

In [ ]:
# Save soft prompt checkpoint to GitHub
# Run this after Phase 3 completes so you can resume from Phase 4 in a new session
from getpass import getpass
from pathlib import Path
import os

CODE_DIR = os.environ.get('CODE_DIR', '/content/xauusd-llm-forecaster')
env_name = 'Kaggle' if os.path.exists('/kaggle') else 'Colab'

sp_path = Path(f'{CODE_DIR}/checkpoints/soft_prompt/best_soft_prompt.npy')
if sp_path.exists():
    token = getpass('GitHub PAT (hidden): ')
    !git -C {CODE_DIR} config user.email "training@env"
    !git -C {CODE_DIR} config user.name "{env_name}"
    !git -C {CODE_DIR} remote set-url origin https://{token}@github.com/211abhi/xauusd-llm-forecaster.git
    !git -C {CODE_DIR} stash
    !git -C {CODE_DIR} pull --rebase origin main
    !git -C {CODE_DIR} add -f checkpoints/soft_prompt/best_soft_prompt.npy
    !git -C {CODE_DIR} commit -m "Save trained soft prompt checkpoint"
    !git -C {CODE_DIR} push origin main
    print('Soft prompt pushed to GitHub.')
else:
    print('No soft prompt checkpoint found.')

## Phase 4 — Train Prediction Head

Trains only the 2-layer MLP on frozen encoder + frozen LLM + frozen soft prompt.

⏱ ~5–15 min on T4.

In [ ]:
!python {CODE_DIR}/scripts/05_train_pred_head.py --config {CODE_DIR}/configs/base_config.yaml

In [ ]:
from pathlib import Path
import os
CODE_DIR = os.environ.get('CODE_DIR', '/content/xauusd-llm-forecaster')
ph_path = Path(f'{CODE_DIR}/checkpoints/pred_head/best_pred_head.pt')
print('Pred head saved:', ph_path.exists())

In [ ]:
# Save pred head checkpoint to GitHub
from getpass import getpass
from pathlib import Path
import os

CODE_DIR = os.environ.get('CODE_DIR', '/content/xauusd-llm-forecaster')
env_name = 'Kaggle' if os.path.exists('/kaggle') else 'Colab'

ph_path = Path(f'{CODE_DIR}/checkpoints/pred_head/best_pred_head.pt')
if ph_path.exists():
    token = getpass('GitHub PAT (hidden): ')
    !git -C {CODE_DIR} config user.email "training@env"
    !git -C {CODE_DIR} config user.name "{env_name}"
    !git -C {CODE_DIR} remote set-url origin https://{token}@github.com/211abhi/xauusd-llm-forecaster.git
    !git -C {CODE_DIR} stash
    !git -C {CODE_DIR} pull --rebase origin main
    !git -C {CODE_DIR} add -f checkpoints/pred_head/best_pred_head.pt
    !git -C {CODE_DIR} commit -m "Save trained pred head checkpoint"
    !git -C {CODE_DIR} push origin main
    print('Pred head pushed to GitHub.')
else:
    print('No pred head checkpoint found.')

## Phase 5 — Evaluation

Runs the full pipeline on the held-out test set and reports MAE, RMSE, and directional accuracy.

In [ ]:
!python {CODE_DIR}/scripts/06_evaluate.py --config {CODE_DIR}/configs/base_config.yaml

In [ ]:
import json, os
CODE_DIR = os.environ.get('CODE_DIR', '/content/xauusd-llm-forecaster')
with open(f'{CODE_DIR}/logs/evaluation_results.json') as f:
    results = json.load(f)

print('=== Overall ===')
for k, v in results['overall'].items():
    print(f'  {k}: {v:.4f}')

if 'per_regime' in results:
    print('\n=== Per Regime ===')
    for regime, m in sorted(results['per_regime'].items()):
        print(f"  {regime:15s}  MAE={m['mae']:.4f}  DirAcc={m['directional_accuracy']:.1f}%")

## Quick Inference — Single Prediction

Run this after all phases are complete to test the forecaster on a single window.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, yaml, os
from src.prediction.forecaster import Forecaster
from src.utils.data_loader import get_feature_columns

CODE_DIR = os.environ.get('CODE_DIR', '/content/xauusd-llm-forecaster')
with open(f'{CODE_DIR}/configs/base_config.yaml') as f:
    cfg = yaml.safe_load(f)

forecaster = Forecaster.from_config(cfg)

# Load normalized features for model input
df_proc = pd.read_csv(f'{CODE_DIR}/data/processed/xau_with_regimes.csv', parse_dates=['datetime'])
# Load raw prices for denormalization and display
df_raw  = pd.read_csv(f'{CODE_DIR}/data/raw/XAU_30m_data.csv', sep=';', parse_dates=['datetime'])
df_raw.columns = [c.lower().strip() for c in df_raw.columns]
df_raw = df_raw.sort_values('datetime').reset_index(drop=True)

feature_cols = get_feature_columns(cfg)
arr = df_proc[feature_cols].values

W        = cfg['tokenizer']['window_size']
n_steps  = cfg['prediction_head']['output_steps']
norm_win = cfg['data']['norm_window']

# ── Predict on last window of test set ────────────────────────
window      = arr[-W - n_steps: -n_steps][np.newaxis]  # (1, W, F)
preds_norm  = forecaster.predict(window)[0]             # (n_steps,) normalized

# ── Denormalize using raw close min/max ───────────────────────
raw_close   = df_raw['close'].values.astype(float)
pred_start  = len(raw_close) - n_steps
norm_slice  = raw_close[max(0, pred_start - norm_win): pred_start]
close_min, close_max = norm_slice.min(), norm_slice.max()
price_range = close_max - close_min

preds_price  = preds_norm  * price_range + close_min
actual_price = raw_close[-n_steps:]

# ── Historical context (last 48 candles = 24 hours) ───────────
context_len  = 48
hist_price   = raw_close[-n_steps - context_len: -n_steps]
hist_times   = np.arange(-context_len, 0)
pred_times   = np.arange(0, n_steps)

# ── Plot ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(hist_times, hist_price, color='gray', linewidth=1.2, label='Historical (24 h)')
ax.plot(pred_times, actual_price, 'o-', color='steelblue', linewidth=1.5, label='Actual')
ax.plot(pred_times, preds_price,  's--', color='tomato',    linewidth=1.5, label='Predicted')
ax.axvline(0, color='black', linestyle=':', linewidth=0.8, label='Forecast start')
ax.fill_between(pred_times, preds_price, actual_price, alpha=0.12, color='tomato')
ax.set_title(f'XAU/USD — {n_steps}-Step Forecast ({n_steps * 30 // 60}h ahead, 30-min bars)')
ax.set_xlabel('Step (×30 min)')
ax.set_ylabel('Price (USD)')
ax.legend()
plt.tight_layout()
plt.show()

mae   = abs(preds_price - actual_price).mean()
rmse  = np.sqrt(((preds_price - actual_price) ** 2).mean())
dacc  = (np.sign(np.diff(preds_price)) == np.sign(np.diff(actual_price))).mean() * 100
print(f'MAE  : ${mae:.2f}')
print(f'RMSE : ${rmse:.2f}')
print(f'Dir  : {dacc:.1f}%')
print(f'Pred : ${preds_price.min():.2f} – ${preds_price.max():.2f}')
print(f'Act  : ${actual_price.min():.2f} – ${actual_price.max():.2f}')

## Save Checkpoints
Download trained weights before the session ends.

In [ ]:
import zipfile, os
CODE_DIR = os.environ.get('CODE_DIR', '/content/xauusd-llm-forecaster')
is_kaggle = os.path.exists('/kaggle')

zip_dir = '/kaggle/working' if is_kaggle else '/content'
zip_path = f'{zip_dir}/checkpoints.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for root, _, filenames in os.walk(f'{CODE_DIR}/checkpoints'):
        for fname in filenames:
            fpath = os.path.join(root, fname)
            zf.write(fpath, os.path.relpath(fpath, CODE_DIR))

print(f'Zipped checkpoints → {zip_path}')
if is_kaggle:
    print('Kaggle: find checkpoints.zip in the Output tab on the right panel.')
else:
    from google.colab import files
    files.download(zip_path)